##### ***从零构建GPT-2***
###### 该教程基于YouTube上的大佬Adrej Karpathy的视频Deep Dive into LLMs like ChatGPT，视频链接如下：https://www.youtube.com/watch?v=7xTGNNLPyMI&t=183s
###### 在学习了BERT之后，我们以及理解了pretraining模型并不只是简单的为为一个词分配一个固定向量，而是通过上下文来学习词元表示。只不过由于BERT模型的架构导致其更适合理解类任务而不擅长生生任务，所以另一条路线就是GPT系列模型。具体来说，BERT模型由于双向建模的原因使得它们能同时理解上下文，这对于填空类或理解类的任务十分友好，而生成式模型往往只能根据前面的信息生成后面的信息，所以BERT模型在生成式任务上的表现很糟糕，进而提出了GPT系列的模型，根据前文预测接下来的token。
###### 接下来我们从头开始一步步的回顾前文知识并构建GPT-2。
##### ***Step1 获取训练数据***
###### 由于LLMs的训练需要大量数据，所以传统的方法不太适应，但由于LLMs的性质，它并不需要像类似于Image的数据进行标注，它只需要大量的文本数据。所以高效获取数据的手段之一就是爬虫，通过爬虫技术爬取大量的HTML文件然后将其按某种规则进行过滤处理，得到想要的文本数据。
##### ***Step2 Tokenizer***
###### 当获得数据后并不能像图像分类模型那样直接将原始数据作为输入，因为图片本质上是像素点构成，通过不同的像素值可以得到一定的关系形式，而文本数据由于不同的语言，语法，逻辑等不是计算机能直接理解的形式，所以需要对其进行处理，将其从自然语言字符串转化为数字张量。所以有了tokenizer这一步，通过将原始文本切分成更小的单元Token构成一个词表，比如单个字符或者单词，或者子词，或者词组，或者前后缀等多种形式。但一般不直接按照单词进行切分，因为自然语言中的单词非常之多，而且各种相似的变体也让这种切分方法的效率极低，开销极大，所以现代模型更倾向于子词级别的分词方式，如之前学过的BPE，字节对编码，通过合并出现最频繁的字符对，来大幅减少词表的总量。进行tokenizer后原始文本就会被转换为一串Token ID，于是原始文本便成了一连串的ID编号，但它仍是离散的编号还是不能被神经网络处理。
##### ***Step3 Embedding Layer***
###### 经过tokenizer的数据变成了一个个对应的Token ID，这种ID只是能表示原始数据的唯一性但是它难以表示不同编号的差异，也难以表示语义，并且如果直接把Token ID输入，由于各个ID的大小不同，这种大小会误导神经网络。于是在正式将数据送入神经网络之前，我们需要对其进一步的处理，Embedding Layer便是其处理方式，它将这些Token ID映射成向量表示，于是经过Embedding Layer的数据将会变成一个形状为（vocab_size, num_hidden）的词嵌入矩阵，这样一来，之前的每一个Token ID将会对应每一行。而具体如何将Token ID转为词嵌入向量见前文的各类算法Word2vec，Glove等。但仅有Token Embedding还不够，为了让模型理解Token的顺序，还需要添加一个Position Embedding，但语言的顺序对于理解语义来说十分重要，于是在Token Embedding矩阵上还要再加一个Position Embedding矩阵。
##### ***Step4 Transformer***
###### 经过Embedding Layer后的数据会变成一个形状为（batch_size, seq_len, num_hidden）的矩阵，这里的seq_len指当前序列的长度，num_hidden指隐藏向量维度，此时数据已经可以转化为连续的向量表示了，接下来输入进神经网络中，而对于LLM来说，现在的神经网络框架基本都是Transformer及其变体。区别于BERT双向注意力的结构，GPT-2只使用了解码器，也就是每个token只能看到它之前的token，而看不到未来的token，这样的结构正好适用于语言生成任务。而GPT-2的解码器块与原始Transformer块的结构略有不同，但其核心组件一致。即：首先将输入序列输入带掩码的多头自注意力块中，让模型从不同角度理解语义，得到K、Q、V，通过K、Q、V计算注意力分数$scores = \frac{Q @ K^T }{\sqrt{d_k}}$。接着应用LayerNorm有助于训练稳定性，但在GPT-2中使用的是pre-LN，也就是说整个GPT-2的Transformer块的结构是Input->Pre_LN->Attention->Res_Connect->LN->FFN->Res_Connect，多个这样的块串行后将输出送到MLP得到最终输出。
###### 到这里为止，GPT-2的模型结构主干就已经基本确定了。也就是说，原始文本先经过tokenizer转化为Token ID，再通过Embedding Layer与Position Embedding变成输入表示，随后送入多个Transformer块中完成上下文建模。接下来剩下的工作，更多就是定义输出层、构造next-token prediction的训练目标，并通过反向传播不断更新整个模型参数。于是，后续我们将直接通过代码来一步步实现这一过程。


##### ***Step5 代码实现：最小GPT-2骨架***
###### 接下来先不急着上完整训练流程，而是先实现一个最小可运行的GPT-2骨架。之所以这样做，是因为对于初学者来说，最难的往往不是理论本身，而是如何把理论拆成一个个可以运行的模块。因此我们先按照`Config -> Attention -> MLP -> Block -> GPT`这样的顺序把模型主干搭起来，并用一个随机输入检查前向传播是否正常。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

###### 首先定义一个配置类来统一管理模型超参数，例如词表大小、最大序列长度、层数、注意力头数以及隐藏维度。这样做的好处是，后面每个模块都只需要接收一个`config`对象，而不需要手动传很多零散参数。

In [2]:
class GPTConfig():
    def __init__(self):
        self.vocab_size = 100000
        self.max_len = 256
        self.num_embd = 512 # embedding层的维度要能被num_heads整除
        self.num_heads = 4
        self.num_layer = 6
        self.dropout = 0.1

###### 自注意力模块是GPT-2中最关键的部分。这里我们一次性通过一个线性层得到Q、K、V，再按照多头注意力的方式拆成多个head。随后利用下三角掩码屏蔽未来位置，确保每个token只能看到自己和前面的token，这就是GPT-2实现自回归生成的关键。

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.num_embd % config.num_heads == 0 # 先进行逻辑检查
        self.num_embd = config.num_embd
        self.num_heads = config.num_heads
        self.num_dim = config.num_embd // config.num_heads

        # GPT-2中的Attention机制是只做一次线性映射，直接把输入投影成3倍维度，然后再切成K、Q、V
        self.c_attn = nn.Linear(config.num_embd, 3 * config.num_embd)
        self.c_proj = nn.Linear(config.num_embd, config.num_embd) # 这里通过线性层把多头信息重新混合

        # 这里通常有两个dropout，一个是加在Attention权重上的，一个是加在输出投影上的。
        self.attn_dropout = nn.Dropout(config.dropout)
        self.proj_dropout = nn.Dropout(config.dropout)

        # 为了实现因果掩码的效果，我们需要一个下三角矩阵，即一个矩阵在对角线下的部分是可以填写数值的，上半部分 -inf
        # 这里使用register_buffer是因为我们希望这个矩阵能一起放进GPU但是不参与梯度计算
        self.register_buffer("causal_mask", 
        torch.tril(torch.ones(config.max_len, config.max_len)).view(1, 1, config.max_len, config.max_len)) #由于维度原因，注意力分数的维度通常是(batch_size, num_heads, T, T)四维的，所以这里我们使用view将这个矩阵也变成四维的，后续前两个维度将会自动广播。

    def forward(self, x):
        B, T, C = x.shape # batch_size, seq_len, embedding_dim
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.num_embd, dim=2)

        q = q.view(B, T, self.num_heads, self.num_dim).transpose(1, 2) # 交换第一二维度，方便后续注意力分数的计算
        k = k.view(B, T, self.num_heads, self.num_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.num_dim).transpose(1, 2)

        attention = (q @ k.transpose(-2, -1)) * (1.0 / (self.num_dim ** 0.5))
        attention = attention.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        attention = F.softmax(attention, dim=-1)
        attention = self.attn_dropout(attention)

        y = attention @ v # (B, num_heads, T, num_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # 这里由于需要将维度合并回去，所以需要contiguous来保证内存的连续性。
        y = self.c_proj(y)
        y = self.proj_dropout(y)
        return y

###### 前馈网络MLP负责在每个位置上进一步做非线性变换，而Block则把LayerNorm、自注意力、残差连接和MLP组合在了一起。这里采用的是GPT-2常见的pre-LN结构，也就是先做LayerNorm，再进入attention或MLP模块。

In [4]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.num_embd, 4 * config.num_embd)
        self.c_proj = nn.Linear(4 * config.num_embd, config.num_embd)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

In [5]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.num_embd)
        self.attention = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.num_embd)
        self.mlp = MLP(config)
    
    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [6]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.num_embd)
        self.position_embed = nn.Parameter(torch.zeros(1, config.max_len, config.num_embd))
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.num_layer)])
        self.ln_f = nn.LayerNorm(config.num_embd)
        self.lm_head = nn.Linear(config.num_embd, config.vocab_size, bias=False)

    def forward(self, token_idx, targets=None):
        B, T = token_idx.shape
        assert T <= self.config.max_len
        token_embd = self.token_embedding(token_idx)
        position_embed = self.position_embed[:, :T, :]
        x = self.dropout(token_embd + position_embed)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


###### 到这一步为止，我们已经把GPT-2最核心的主干拼起来了：输入先经过token embedding和position embedding，再通过多个Transformer Block完成上下文建模，最后通过线性层映射回词表大小，得到每个位置关于下一个token的预测分数。如果传入`targets`，这里还会顺手计算next-token prediction对应的交叉熵损失。

In [7]:
config = GPTConfig()
model = GPT(config)
x = torch.randint(0, config.vocab_size, (2, 16))
y = torch.randint(0, config.vocab_size, (2, 16))

logits, loss = model(x, y)
print("x shape:", x.shape)
print("y shape:", y.shape)
print("logits shape:", logits.shape)
print("loss:", loss)

x shape: torch.Size([2, 16])
y shape: torch.Size([2, 16])
logits shape: torch.Size([2, 16, 100000])
loss: tensor(11.5458, grad_fn=<NllLossBackward0>)
